# Image Enhancer CNN

In [1]:
%%capture
!pip install onnxruntime

In [2]:
import csv
import json
import os
from pathlib import Path

import numpy as np
import onnx
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

import subprocess
import sys
import shutil
import onnx
import onnxruntime as ort

SEED = 42
IMAGE_SIZE = 384
BATCH_SIZE = 32
EPOCHS = 30
LR = 1e-3
NUM_WORKERS = 0
PARAMS = ["brightness", "contrast", "saturation"]
COLOR_SCHEME = "RGB"
INPUT_NAMES = ["image", "stats"]
OUTPUT_NAME = "params"
LEVELS = ["small", "high"]
TRAIN_DISTORTION_LEVELS = ["small", "high"]
STATS_SIZE = 18


IS_KAGGLE = "KAGGLE_URL_BASE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/datasets/marselberheev/corrupted/processed")
    MODEL_DIR = Path("/kaggle/working/models")
else:
    NOTEBOOK_DIR = Path.cwd()
    if NOTEBOOK_DIR.name == "notebooks":
        PROJECT_ROOT = NOTEBOOK_DIR.parent
    else:
        PROJECT_ROOT = NOTEBOOK_DIR

    DATA_DIR = PROJECT_ROOT / "data" / "processed"
    MODEL_DIR = PROJECT_ROOT / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

LABELS_CSV = DATA_DIR / "labels.csv"
SPLIT_DIRS = {split: DATA_DIR / split for split in ("train", "val", "test")}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Device:      {DEVICE}")
print(f"Data dir:    {DATA_DIR}")
print(f"Model dir:   {MODEL_DIR}")
print(f"Train distortion levels: {TRAIN_DISTORTION_LEVELS}")

Device:      cuda
Data dir:    /kaggle/input/datasets/marselberheev/corrupted/processed
Model dir:   /kaggle/working/models
Train distortion levels: ['small', 'high']


In [3]:
def image_stats(image):
    flat = image.flatten(1)
    channel_mean = flat.mean(dim=1)
    channel_std = flat.std(dim=1, unbiased=False)
    channel_var = flat.var(dim=1, unbiased=False)
    channel_min = flat.min(dim=1).values
    channel_max = flat.max(dim=1).values
    global_flat = image.flatten()
    global_stats = torch.stack([
        global_flat.mean(),
        global_flat.std(unbiased=False),
        global_flat.var(unbiased=False),
    ])
    return torch.cat([channel_mean, channel_std, channel_var, channel_min, channel_max, global_stats]).float()


class EnhancementDataset(Dataset):
    def __init__(self, split, levels=None):
        self.split = split
        self.levels = set(levels or LEVELS)
        unknown = self.levels - set(LEVELS)
        if unknown:
            raise ValueError(f"Unknown distortion levels: {sorted(unknown)}")

        with LABELS_CSV.open("r", newline="", encoding="utf-8") as file:
            rows = list(csv.DictReader(file))

        self.rows = [
            row for row in rows
            if row["image_id"].startswith(f"{split}_") and row["distortion_level"] in self.levels
        ]
        if not self.rows:
            raise ValueError(f"No {split} samples for levels {sorted(self.levels)}")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        path = SPLIT_DIRS[self.split] / f"{row['image_id']}.jpg"
        image = Image.open(path).convert(COLOR_SCHEME).resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
        image = torch.from_numpy(np.asarray(image, dtype=np.float32) / 255.0).permute(2, 0, 1)
        stats = image_stats(image)
        target = torch.tensor([float(row[name]) for name in PARAMS], dtype=torch.float32)
        level = torch.tensor(LEVELS.index(row["distortion_level"]), dtype=torch.long)
        return image, stats, target, level


def loader(split, levels=None, shuffle=False):
    dataset = EnhancementDataset(split, levels)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))


train_loader = loader("train", TRAIN_DISTORTION_LEVELS, shuffle=True)
val_loader = loader("val")
test_loader = loader("test")

print(f"Samples: train={len(train_loader.dataset)} val={len(val_loader.dataset)} test={len(test_loader.dataset)}")
image, stats, target, level = next(iter(train_loader))
image.shape, stats.shape, target.shape, level.shape

Samples: train=28800 val=3600 test=3600


(torch.Size([32, 3, 384, 384]),
 torch.Size([32, 18]),
 torch.Size([32, 3]),
 torch.Size([32]))

In [4]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.se(x)


class EfficientBlock(nn.Module):
    """Depthwise separable conv + optional residual."""
    def __init__(self, in_ch, out_ch, stride):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, 3, stride, padding=1, groups=in_ch, bias=False)
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.se = SEBlock(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.use_res = (stride == 1 and in_ch == out_ch)

    def forward(self, x):
        out = self.relu(self.bn1(self.dw(x)))
        out = self.bn2(self.pw(out))
        out = self.se(out)
        if self.use_res:
            out = out + x
        return self.relu(out)


class LightEnhancementCNN(nn.Module):
    """Efficient multi‑scale CNN for brightness / saturation / contrast prediction."""
    def __init__(self, stats_dim=STATS_SIZE):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1))

        self.stage1 = nn.Sequential(
            EfficientBlock(3, 16, stride=2),
            EfficientBlock(16, 16, stride=1)
        )
        self.stage2 = nn.Sequential(
            EfficientBlock(16, 32, stride=2),
            EfficientBlock(32, 32, stride=1)
        )
        self.stage3 = nn.Sequential(
            EfficientBlock(32, 64, stride=2),
            EfficientBlock(64, 64, stride=1)
        )
        self.stage4 = nn.Sequential(
            EfficientBlock(64, 96, stride=2),
            EfficientBlock(96, 96, stride=1)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)

        head_in = 16 + 32 + 64 + 96 + stats_dim
        self.head = nn.Sequential(
            nn.Linear(head_in, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.15),
            nn.Linear(64, 3)
        )

    def forward(self, image, stats):
        image = (image - self.mean) / self.std

        s1 = self.stage1(image)     # [B,16, H/2, W/2]
        s2 = self.stage2(s1)        # [B,32, H/4, W/4]
        s3 = self.stage3(s2)        # [B,64, H/8, W/8]
        s4 = self.stage4(s3)        # [B,96, H/16, W/16]

        f1 = self.pool(s1).flatten(1)
        f2 = self.pool(s2).flatten(1)
        f3 = self.pool(s3).flatten(1)
        f4 = self.pool(s4).flatten(1)

        combined = torch.cat([f1, f2, f3, f4, stats], dim=1)
        raw = self.head(combined)

        return torch.cat([
            raw[:, 0:1],
            torch.exp(raw[:, 1:2]),
            torch.exp(raw[:, 2:3])
        ], dim=1)

model = LightEnhancementCNN(stats_dim=STATS_SIZE).to(DEVICE)
print(model(torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE),
            torch.zeros(1, STATS_SIZE, device=DEVICE)).shape)

torch.Size([1, 3])


In [5]:
PARAM_WEIGHTS = torch.tensor([2.0, 1.0, 1.5], device=DEVICE)


def prepare_targets_for_loss(target):
    """Convert targets to log‑space for saturation & contrast, keep brightness linear."""
    brightness = target[:, 0:1]
    sat = target[:, 1:2].clamp_min(1e-4)
    cont = target[:, 2:3].clamp_min(1e-4)
    return torch.cat([brightness, torch.log(sat), torch.log(cont)], dim=1)


def run_epoch(model, data, optimizer=None):
    training = optimizer is not None
    model.train(training)
    loss_fn = nn.SmoothL1Loss(reduction="none")

    total = 0.0
    per_param = torch.zeros(3, device=DEVICE)
    per_level = torch.zeros(len(LEVELS), device=DEVICE)
    per_level_param = torch.zeros(len(LEVELS), 3, device=DEVICE)
    level_count = torch.zeros(len(LEVELS), device=DEVICE)
    count = 0

    with torch.set_grad_enabled(training):
        for image, stats, target, level in data:
            image  = image.to(DEVICE)
            stats  = stats.to(DEVICE)
            target = target.to(DEVICE)
            level  = level.to(DEVICE)

            # [brightness, saturation, contrast]
            pred_factors = model(image, stats)

            pred_loss = torch.cat([
                pred_factors[:, 0:1],
                torch.log(pred_factors[:, 1:2].clamp_min(1e-4)),
                torch.log(pred_factors[:, 2:3].clamp_min(1e-4))
            ], dim=1)

            target_loss = prepare_targets_for_loss(target)

            param_loss_raw = loss_fn(pred_loss, target_loss)
            param_loss = param_loss_raw * PARAM_WEIGHTS
            sample_loss = param_loss.mean(dim=1)
            loss = sample_loss.mean()

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total += sample_loss.detach().sum().item()
            per_param += param_loss_raw.detach().sum(dim=0)
            count += image.size(0)

            for level_idx in range(len(LEVELS)):
                mask = level == level_idx
                if mask.any():
                    per_level[level_idx] += sample_loss.detach()[mask].sum()
                    per_level_param[level_idx] += param_loss_raw.detach()[mask].sum(dim=0)
                    level_count[level_idx] += mask.sum()

    metrics = {
        "total": total / count,
        "param": (per_param / count).cpu().tolist(),
        "level": {},
    }
    for i, name in enumerate(LEVELS):
        if level_count[i] > 0:
            metrics["level"][name] = {
                "total": (per_level[i] / level_count[i]).item(),
                "param": (per_level_param[i] / level_count[i]).cpu().tolist(),
            }
    return metrics


def metric_table(title, metrics_by_split):
    """Display loss table – no MAPE."""
    rows = [f"\n{title}", "split    level      total   brightness   contrast   saturation"]
    rows.append("-" * len(rows[-1]))
    for split, metrics in metrics_by_split.items():
        p = metrics["param"]
        rows.append(f"{split:<8} {'all':<7} {metrics['total']:>8.5f} {p[0]:>12.5f} {p[1]:>10.5f} {p[2]:>12.5f}")
        for level in LEVELS:
            values = metrics["level"].get(level)
            if values is None:
                continue
            lp = values["param"]
            rows.append(f"{split:<8} {level:<7} {values['total']:>8.5f} {lp[0]:>12.5f} {lp[1]:>10.5f} {lp[2]:>12.5f}")
    return "\n".join(rows)


def train_model(model):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    best_state, best_val = None, float("inf")

    for epoch in tqdm(range(EPOCHS), desc="Training"):
        train_metrics = run_epoch(model, train_loader, optimizer)
        val_metrics = run_epoch(model, val_loader)

        if val_metrics["total"] < best_val:
            best_val = val_metrics["total"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step()
        tqdm.write(metric_table(f"Epoch {epoch + 1:02d}", {"train": train_metrics, "val": val_metrics}))

    model.load_state_dict(best_state)
    return model

model = train_model(model)

test_metrics = run_epoch(model, test_loader)
print(metric_table("Test", {"test": test_metrics}))

Training:   0%|          | 0/30 [00:00<?, ?it/s]


Epoch 01
split    level      total   brightness   contrast   saturation
--------------------------------------------------------------
train    all      0.04569      0.00532    0.02918      0.06482
train    small    0.01417      0.00404    0.01451      0.01328
train    high     0.05781      0.00581    0.03483      0.08465
val      all      0.03903      0.00342    0.02353      0.05782
val      small    0.01016      0.00151    0.01164      0.01055
val      high     0.05014      0.00415    0.02810      0.07600

Epoch 02
split    level      total   brightness   contrast   saturation
--------------------------------------------------------------
train    all      0.03828      0.00356    0.02341      0.05622
train    small    0.01204      0.00231    0.01099      0.01366
train    high     0.04837      0.00403    0.02818      0.07258
val      all      0.03476      0.00233    0.01875      0.05391
val      small    0.01150      0.00221    0.00909      0.01399
val      high     0.04371      0.00

In [6]:
def next_model_dir(root):
    root.mkdir(parents=True, exist_ok=True)
    used_numbers = []
    for path in root.glob("model_*"):
        if not path.is_dir():
            continue
        try:
            used_numbers.append(int(path.name.rsplit("_", 1)[1]))
        except ValueError:
            pass
    next_number = max(used_numbers, default=0) + 1
    return root / f"model_{next_number:02d}"


def model_config():
    return {
        "input_image_size": IMAGE_SIZE,
        "color_scheme": COLOR_SCHEME,
        "input_param_names": INPUT_NAMES,
        "output_name": OUTPUT_NAME,
        "output_param_names": PARAMS,
    }

In [7]:
def export_model(model, model_dir):
    """Export ONNX, ORT variants, config and required ops – using the ORT CLI."""
    model.eval().cpu()
    model_dir.mkdir(parents=True, exist_ok=True)

    onnx_path = model_dir / "model.onnx"
    config_path = model_dir / "config.json"
    req_ops_path = model_dir / "model.required_operators.config"
    req_ops_opt_path = model_dir / "model.required_operators.with_runtime_opt.config"

    dummy_image = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)
    dummy_stats = torch.zeros(1, STATS_SIZE, dtype=torch.float32)

    torch.onnx.export(
        model,
        (dummy_image, dummy_stats),
        str(onnx_path),
        input_names=INPUT_NAMES,
        output_names=[OUTPUT_NAME],
        opset_version=17,
        do_constant_folding=True,
        dynamo=False,
    )
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)

    config_path.write_text(json.dumps(model_config(), indent=2) + "\n", encoding="utf-8")

    opset_imports = {domain.domain: domain.version for domain in onnx_model.opset_import}
    op_types = sorted({node.op_type for node in onnx_model.graph.node})
    req = {"opset": opset_imports, "operators": op_types}
    req_ops_path.write_text(json.dumps(req, indent=2) + "\n", encoding="utf-8")
    req_ops_opt_path.write_text(json.dumps(req, indent=2) + "\n", encoding="utf-8")

    python_exe = sys.executable
    base_cmd = [python_exe, "-m", "onnxruntime.tools.convert_onnx_models_to_ort", str(onnx_path)]

    tmp_static = model_dir / "_tmp_static"
    tmp_static.mkdir(exist_ok=True)
    subprocess.run(base_cmd + ["--output_dir", str(tmp_static), "--optimization_style", "Fixed"], check=True)
    static_ort = tmp_static / "model.ort"
    if static_ort.exists():
        shutil.move(str(static_ort), str(model_dir / "model.ort"))
    shutil.rmtree(tmp_static, ignore_errors=True)

    tmp_runtime = model_dir / "_tmp_runtime"
    tmp_runtime.mkdir(exist_ok=True)
    subprocess.run(base_cmd + ["--output_dir", str(tmp_runtime), "--optimization_style", "Runtime"], check=True)
    runtime_ort = tmp_runtime / "model.ort"
    if runtime_ort.exists():
        shutil.move(str(runtime_ort), str(model_dir / "model.with_runtime_opt.ort"))
    shutil.rmtree(tmp_runtime, ignore_errors=True)

    return model_dir

checkpoint_dir = next_model_dir(MODEL_DIR)
export_model(model, checkpoint_dir)

/tmp/ipykernel_23/3772983457.py:14: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
2026-06-02 02:41:27,369 ort_format_model.utils [INFO] - Created config in /kaggle/working/models/model_01/_tmp_static/model.required_operators.config


Converting models with optimization style 'Fixed' and level 'all'
Converting optimized ONNX model /kaggle/working/models/model_01/model.onnx to ORT format model /kaggle/working/models/model_01/_tmp_static/model.ort
Converted 1/1 models successfully.
Generating config file from ORT format models with optimization style 'Fixed' and level 'all'


2026-06-02 02:41:30,961 ort_format_model.utils [INFO] - Created config in /kaggle/working/models/model_01/_tmp_runtime/model.required_operators.with_runtime_opt.config


Converting models with optimization style 'Runtime' and level 'all'
Converting optimized ONNX model /kaggle/working/models/model_01/model.onnx to ORT format model /kaggle/working/models/model_01/_tmp_runtime/model.with_runtime_opt.ort
Converted 1/1 models successfully.
Converting models again without runtime optimizations to generate a complete config file. These converted models are temporary and will be deleted.
Converting optimized ONNX model /kaggle/working/models/model_01/model.onnx to ORT format model /kaggle/working/models/model_01/tmpbe8mbv0y.without_runtime_opt/model.ort
Converted 1/1 models successfully.
Generating config file from ORT format models with optimization style 'Runtime' and level 'all'


PosixPath('/kaggle/working/models/model_01')